# 페르소나 가상 투표 실험 — 1인 파일럿 (LangChain + OpenAI)

`nvidia/Nemotron-Personas-Korea`에서 1명을 랜덤 샘플링해, 2026-04 한국 정치 상황 컨텍스트를 주고 OpenAI 모델에게 투표 의향과 이유를 받는다.

**저장 컬럼**: `persona_uuid, vote, reason, model, elapsed_sec`

## 0. 사전 준비

```bash
pip install langchain langchain-openai langchain-ollama python-dotenv pyarrow pandas
```

프로젝트 루트 또는 `backend/`에 `.env.local` 생성:
```
OPENAI_API_KEY=sk-...
```

OpenAI 모델은 [Cell 1]의 `MODEL` 변수에서, 로컬(Ollama) 모델은 [Cell 10~12]의 `model_name` 인자에서 직접 지정.

In [ ]:
# [Cell 1] imports & env
import os, glob, json, time, random, re
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 노트북이 있는 폴더 (= backend/) 기준 경로
BASE = Path.cwd() if Path.cwd().name == "backend" else Path.cwd() / "backend"
DATA_DIR = BASE / "nvidia-personas" / "data"
CONTEXT_DIR = BASE / "context"
RESULTS_PATH = BASE / "vote_results_all.csv"
RESPONSE_DIR = BASE / "response"
RESPONSE_DIR.mkdir(exist_ok=True)

# OpenAI 기본 모델 — Cell 9에서 사용
MODEL = "gpt-5.4-mini"

# 컨텍스트 버전 — voter_context_*.md / system_prompt_*.md 항상 같은 v로 페어링
VERSION = "v1"

# .env.local 로드 — backend/.env.local 우선, 없으면 프로젝트 루트
for env_path in [BASE / ".env.local", BASE.parent / ".env.local"]:
    if env_path.exists():
        load_dotenv(env_path)
        print(f"loaded: {env_path}")
        break

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 환경변수 필요 — .env.local 확인"

# 사용 가능한 컨텍스트 버전 자동 감지 (voter·system 둘 다 있는 v만)
def list_versions() -> list[str]:
    voter_pat = re.compile(r"^voter_context_v(\d+)\.md$")
    system_pat = re.compile(r"^system_prompt_v(\d+)\.md$")
    voter_vs = {f"v{voter_pat.match(p.name).group(1)}" for p in CONTEXT_DIR.iterdir() if voter_pat.match(p.name)}
    system_vs = {f"v{system_pat.match(p.name).group(1)}" for p in CONTEXT_DIR.iterdir() if system_pat.match(p.name)}
    return sorted(voter_vs & system_vs, key=lambda v: int(v[1:]))

def safe_filename(s: str) -> str:
    return re.sub(r"[^\w\-.]", "_", s)

print(f"openai model: {MODEL}")
print(f"BASE: {BASE}")
print(f"available versions: {list_versions()}")

In [58]:
# [Cell 2] load voter context + system prompt (선택된 버전, 같은 v로 페어링)
def load_context(version: str) -> str:
    return (CONTEXT_DIR / f"voter_context_{version}.md").read_text(encoding="utf-8")

def load_system_prompt(version: str) -> str:
    return (CONTEXT_DIR / f"system_prompt_{version}.md").read_text(encoding="utf-8")

POLITICAL_CONTEXT = load_context(VERSION)
SYSTEM_TEMPLATE = load_system_prompt(VERSION)
print(f"voter_context_{VERSION}.md ({len(POLITICAL_CONTEXT)} chars)")
print(f"system_prompt_{VERSION}.md ({len(SYSTEM_TEMPLATE)} chars)")
print(f"\n--- POLITICAL_CONTEXT preview ---\n{POLITICAL_CONTEXT[:300]} ...")

voter_context_v1.md (996 chars)
system_prompt_v1.md (471 chars)

--- POLITICAL_CONTEXT preview ---
# 한국 정치 상황 (2026년 4월 시점)

> 일반 시민이 뉴스·일상 대화를 통해 인지할 수 있는 수준의 사실만 정리.
> 시점: 2026년 4월 말. 다음 선거: 2026년 6월 3일 제9회 전국동시지방선거.

---

## 정부 구성

- 대통령: 이재명 (더불어민주당, 2025년 6월 취임)
- 여당: 더불어민주당
- 제1야당: 국민의힘

## 주요 정당

원내·원외 주요 정당들:

- **더불어민주당** — 진보·중도 성향. 현 여당.
- **국민의힘** — 보수 성향. 제1야당.
- **개혁신당** — 중도·보수 성향. ...


In [59]:
# [Cell 3] sample 1 persona
# 9개 샤드 중 1개 랜덤 선택 → 그 안에서 1행 랜덤 샘플
shards = sorted(glob.glob(str(DATA_DIR / "train-*.parquet")))
shard = random.choice(shards)
df = pq.read_table(shard).to_pandas()
row = df.sample(n=1).iloc[0]

persona_uuid = row["uuid"]
print(f"shard:    {Path(shard).name}")
print(f"uuid:     {persona_uuid}")
print(f"성/연령:  {row['sex']} / {row['age']}세")
print(f"지역:     {row['province']} {row['district']}")
print(f"직업:     {row['occupation']}")
print(f"학력:     {row['education_level']}")
print(f"\n[persona]\n{row['persona']}")

shard:    train-00006-of-00009.parquet
uuid:     29331d9d786141098962c13e5a76e8b4
성/연령:  여자 / 60세
지역:     전북 전북-순창군
직업:     무직
학력:     중학교

[persona]
강희정 씨는 순창의 대가족을 이끄는 살림의 고수이자, 역사 답사와 스마트한 취미 생활을 통해 제2의 인생을 설계하는 60대 여성입니다.


In [60]:
# [Cell 4] persona card builder
def persona_card(r) -> str:
    fields = [
        ("성별", r["sex"]), ("연령", f"{r['age']}세"),
        ("혼인상태", r["marital_status"]),
        ("가구형태", r["family_type"]), ("주거", r["housing_type"]),
        ("학력", r["education_level"]), ("전공", r["bachelors_field"]),
        ("직업", r["occupation"]),
        ("지역", f"{r['province']} {r['district']}"),
    ]
    demo = "\n".join(f"- {k}: {v}" for k, v in fields if v)
    narr = (
        f"[요약]\n{r['persona']}\n\n"
        f"[직업적 면모]\n{r['professional_persona']}\n\n"
        f"[가족 면모]\n{r['family_persona']}\n\n"
        f"[문화적 배경]\n{r['cultural_background']}\n\n"
        f"[관심사]\n{r['hobbies_and_interests']}\n\n"
        f"[목표]\n{r['career_goals_and_ambitions']}"
    )
    return f"## 인구통계\n{demo}\n\n## 인물 서사\n{narr}"

card = persona_card(row)
print(card[:800], "...")

## 인구통계
- 성별: 여자
- 연령: 60세
- 혼인상태: 배우자있음
- 가구형태: 배우자·자녀·부모와 거주
- 주거: 아파트
- 학력: 중학교
- 전공: 해당없음
- 직업: 무직
- 지역: 전북 전북-순창군

## 인물 서사
[요약]
강희정 씨는 순창의 대가족을 이끄는 살림의 고수이자, 역사 답사와 스마트한 취미 생활을 통해 제2의 인생을 설계하는 60대 여성입니다.

[직업적 면모]
강희정 씨는 공식적인 직업은 없으나, 시부모님과 자녀까지 아우르는 대가족의 살림을 도맡아 하며 집안의 모든 운영 체계를 관리하는 실질적인 총괄 관리자로 살아왔습니다. 유튜브에서 본 최신 살림 팁을 빠르게 적용해 집안 효율을 높이는 것에 소소한 성취감을 느낍니다.

[가족 면모]
강희정 씨는 부모님과 배우자, 그리고 자녀들이 함께 사는 복작복작한 집안의 중심축으로서, 모두의 식성과 컨디션을 세밀하게 챙기는 헌신적인 어머니이자 며느리입니다. 하지만 가끔은 가족들의 사소한 무심함에 서운함을 느껴 갑자기 입을 꾹 닫아버리는 예민한 구석이 있습니다.

[문화적 배경]
강희정 씨는 순창의 고추장 마을 인근에서 나고 자라며 이웃 간의 끈끈한 정과 공동체 문화 속에 살아왔으며, 중학교 졸업 후 일찍이 가정을 꾸려 시부모님과 자녀들을 함께 보살피는 대가족의 중심축 역할을 해왔습니다. 전북 지역 특유의 넉넉한 인심을 체득하면서도, 시대의 변화에 발맞춰 스마트폰으로 새로운 정보를 검색하고 받아들이는 것에 거부감이 없는 유연한 면모를 지니고 있습니다.

[관심사]
주말이면 친한 언니들과 함께 강천산 산책로를 천천히 걷거나 동네 찜질방에서 땀을  ...


In [61]:
# [Cell 5] prompt template & chain builder (provider toggle, 단일 버전)
USER_TEMPLATE = """다음 페르소나가 2026년 6월 3일 지방선거(또는 가까운 미래의 총선·대선)에서 어느 정당을 지지·투표할지 추론하시오.

{persona_card}

JSON만 출력."""

def build_prompt(system_template: str, model_name: str = "") -> ChatPromptTemplate:
    user_tmpl = USER_TEMPLATE
    if "qwen3" in model_name.lower():
        user_tmpl = USER_TEMPLATE + "\n\n/no_think"
    return ChatPromptTemplate.from_messages([
        ("system", system_template),
        ("user", user_tmpl),
    ])

VOTE_KEYS = ("vote", "party", "정당", "지지정당", "투표", "선택", "지지")
REASON_KEYS = ("reason", "이유", "rationale", "사유", "근거", "설명")

def _try_complete_json(s: str) -> str:
    """잘린 JSON에 닫는 따옴표·괄호 보강 시도."""
    # 마지막 } 위치
    last_close = s.rfind("}")
    last_open = s.rfind("{")
    if last_open == -1:
        return s
    if last_close > last_open:
        return s[last_open:last_close + 1]
    # 닫는 } 없음 — 끝에 따옴표·괄호 보강
    candidate = s[last_open:].rstrip().rstrip(",")
    # 짝 안 맞는 따옴표 닫기
    if candidate.count('"') % 2 == 1:
        candidate += '"'
    candidate += "}"
    return candidate

def parse_response(raw: str) -> dict:
    """thinking/코드블록 제거 후 JSON 추출. 잘린 응답은 자동 복구 시도."""
    s = raw
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL)
    s = re.sub(r"```(?:json)?\s*", "", s)
    s = s.replace("```", "")
    # 정상 JSON
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    # 잘린 JSON 복구 시도
    repaired = _try_complete_json(s)
    try:
        return json.loads(repaired)
    except json.JSONDecodeError:
        raise ValueError(f"no parseable JSON in response: {raw[:200]!r}")

def pick(d: dict, keys: tuple) -> str | None:
    for k in keys:
        v = d.get(k)
        if v:
            return v
    return None

def build_chain(provider: str, model_name: str, system_template: str):
    prompt_local = build_prompt(system_template, model_name)
    if provider == "openai":
        llm = ChatOpenAI(
            model=model_name, temperature=0.7,
            model_kwargs={"response_format": {"type": "json_object"}},
        )
    elif provider == "ollama":
        from langchain_ollama import ChatOllama
        llm = ChatOllama(
            model=model_name, temperature=0.7,
            num_ctx=4096, num_predict=1200,  # qwen3 잘림 방지: 800 → 1200
        )
    else:
        raise ValueError(f"unknown provider: {provider}")
    return prompt_local | llm

_chain_cache = {}
def get_chain(provider: str, model_name: str, version: str):
    key = (provider, model_name, version)
    if key not in _chain_cache:
        _chain_cache[key] = build_chain(provider, model_name, load_system_prompt(version))
    return _chain_cache[key]

chain = get_chain("openai", MODEL, VERSION)
print(f"default chain: openai/{MODEL} (version={VERSION})")

default chain: openai/gpt-5.4-mini (version=v1)


In [62]:
# [Cell 6] invoke LLM (디버깅용 — 한 번만 실행)
t0 = time.perf_counter()
msg = chain.invoke({
    "political_context": load_context(VERSION),
    "persona_card": card,
})
elapsed = time.perf_counter() - t0
raw = msg.content if hasattr(msg, "content") else str(msg)
parsed = parse_response(raw)

print(f"[elapsed] {elapsed:.2f}s\n")
print("투표:", parsed["vote"])
print("이유:", parsed["reason"])

[elapsed] 3.48s

투표: 더불어민주당
이유: 저는 전북 순창에서 평생 살아오며 동네 정과 살림살이를 중요하게 여겨왔어요. 지금은 대통령도 민주당이고, 호남에서는 늘 민주당이 지역을 챙겨줄 거라는 믿음이 큽니다. 의료 갈등이나 경제 걱정도 있지만, 적어도 우리 같은 서민 살림과 지역 발전을 더 잘 챙길 정당은 민주당이라고 생각해서 그쪽에 표를 주려 합니다.


In [ ]:
# [Cell 7] save result (디버깅용)
ts = time.strftime("%Y%m%d-%H%M%S")
fname = f"{ts}_openai_{safe_filename(MODEL)}_{persona_uuid[:8]}_{VERSION}_{VERSION}.json"
out = {
    "timestamp": ts,
    "model": f"openai/{MODEL}",
    "voter_context_version": VERSION,
    "system_prompt_version": VERSION,
    "persona_uuid": persona_uuid,
    "elapsed_sec": round(elapsed, 3),
    "persona": {
        "sex": row["sex"], "age": int(row["age"]),
        "marital_status": row["marital_status"],
        "family_type": row["family_type"],
        "housing_type": row["housing_type"],
        "education_level": row["education_level"],
        "bachelors_field": row["bachelors_field"],
        "occupation": row["occupation"],
        "province": row["province"], "district": row["district"],
        "persona_summary": row["persona"],
    },
    "raw_response": raw,
    "parsed": {"vote": parsed["vote"], "reason": parsed["reason"]},
}
(RESPONSE_DIR / fname).write_text(
    json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

record = {
    "persona_uuid": persona_uuid,
    "sex": row["sex"],
    "age": int(row["age"]),
    "marital_status": row["marital_status"],
    "family_type": row["family_type"],
    "housing_type": row["housing_type"],
    "education_level": row["education_level"],
    "bachelors_field": row["bachelors_field"],
    "occupation": row["occupation"],
    "province": row["province"],
    "district": row["district"],
    "persona_summary": row["persona"],
    "vote": parsed["vote"],
    "reason": parsed["reason"],
    "model": f"openai/{MODEL}",
    "elapsed_sec": round(elapsed, 3),
    "response_file": fname,
    "voter_context_version": VERSION,
    "system_prompt_version": VERSION,
    "professional_persona": row["professional_persona"],
    "sports_persona": row["sports_persona"],
    "arts_persona": row["arts_persona"],
    "travel_persona": row["travel_persona"],
    "culinary_persona": row["culinary_persona"],
    "family_persona": row["family_persona"],
    "cultural_background": row["cultural_background"],
    "skills_and_expertise": row["skills_and_expertise"],
    "hobbies_and_interests": row["hobbies_and_interests"],
    "career_goals_and_ambitions": row["career_goals_and_ambitions"],
}

row_df = pd.DataFrame([record])
if RESULTS_PATH.exists():
    row_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")
else:
    row_df.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")

print(f"저장됨 → {RESULTS_PATH}")
print(f"응답 → {RESPONSE_DIR / fname}\n")
pd.read_csv(RESULTS_PATH).tail(3)

In [ ]:
# [Cell 8] one-shot helper — provider/model + version 인자 (단일 버전)
def run_one(provider: str = "openai", model_name: str | None = None,
            version: str = VERSION, verbose: bool = True):
    """1명 샘플 → 투표 추론 → CSV 누적 + response/ JSON 저장"""
    if model_name is None:
        model_name = MODEL
    political_context = load_context(version)
    ch = get_chain(provider, model_name, version)

    # 1) sample
    shard = random.choice(shards)
    df_local = pq.read_table(shard).to_pandas()
    r = df_local.sample(n=1).iloc[0]

    # 2) invoke + parse
    t0 = time.perf_counter()
    raw = ""
    try:
        msg = ch.invoke({
            "political_context": political_context,
            "persona_card": persona_card(r),
        })
        raw = msg.content if hasattr(msg, "content") else str(msg)
        p = parse_response(raw)
        vote = pick(p, VOTE_KEYS)
        reason = pick(p, REASON_KEYS)
        if not vote or not reason:
            raise ValueError(f"missing keys: {list(p.keys())}")
    except Exception as e:
        el = time.perf_counter() - t0
        print(f"[ERROR] ({el:.1f}s) {str(e)[:120]}")
        ts = time.strftime("%Y%m%d-%H%M%S")
        fname = f"FAIL_{ts}_{safe_filename(provider)}_{safe_filename(model_name)}_{r['uuid'][:8]}.txt"
        (RESPONSE_DIR / fname).write_text(
            f"=== ERROR: {e}\n=== MODEL: {provider}/{model_name}\n=== UUID: {r['uuid']}\n\n{raw}",
            encoding="utf-8")
        return None
    el = time.perf_counter() - t0

    # 3) save response JSON
    ts = time.strftime("%Y%m%d-%H%M%S")
    fname = f"{ts}_{safe_filename(provider)}_{safe_filename(model_name)}_{r['uuid'][:8]}_{version}_{version}.json"
    out = {
        "timestamp": ts,
        "model": f"{provider}/{model_name}",
        "voter_context_version": version,
        "system_prompt_version": version,
        "persona_uuid": r["uuid"],
        "elapsed_sec": round(el, 3),
        "persona": {
            "sex": r["sex"], "age": int(r["age"]),
            "marital_status": r["marital_status"],
            "family_type": r["family_type"],
            "housing_type": r["housing_type"],
            "education_level": r["education_level"],
            "bachelors_field": r["bachelors_field"],
            "occupation": r["occupation"],
            "province": r["province"], "district": r["district"],
            "persona_summary": r["persona"],
        },
        "raw_response": raw,
        "parsed": {"vote": vote, "reason": reason},
    }
    (RESPONSE_DIR / fname).write_text(
        json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

    # 4) CSV 누적
    rec = {
        "persona_uuid": r["uuid"],
        "sex": r["sex"], "age": int(r["age"]),
        "marital_status": r["marital_status"],
        "family_type": r["family_type"], "housing_type": r["housing_type"],
        "education_level": r["education_level"],
        "bachelors_field": r["bachelors_field"],
        "occupation": r["occupation"],
        "province": r["province"], "district": r["district"],
        "persona_summary": r["persona"],
        "vote": vote, "reason": reason,
        "model": f"{provider}/{model_name}",
        "elapsed_sec": round(el, 3),
        "response_file": fname,
        "voter_context_version": version,
        "system_prompt_version": version,
        "professional_persona": r["professional_persona"],
        "sports_persona": r["sports_persona"],
        "arts_persona": r["arts_persona"],
        "travel_persona": r["travel_persona"],
        "culinary_persona": r["culinary_persona"],
        "family_persona": r["family_persona"],
        "cultural_background": r["cultural_background"],
        "skills_and_expertise": r["skills_and_expertise"],
        "hobbies_and_interests": r["hobbies_and_interests"],
        "career_goals_and_ambitions": r["career_goals_and_ambitions"],
    }
    df_rec = pd.DataFrame([rec])
    if RESULTS_PATH.exists():
        df_rec.to_csv(RESULTS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_rec.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")

    # 5) pretty print
    if verbose:
        labels = [
            ("uuid",       rec["persona_uuid"]),
            ("성별",        rec["sex"]),
            ("연령",        f'{rec["age"]}세'),
            ("혼인상태",     rec["marital_status"]),
            ("가구형태",     rec["family_type"]),
            ("주거",        rec["housing_type"]),
            ("학력",        rec["education_level"]),
            ("전공",        rec["bachelors_field"]),
            ("직업",        rec["occupation"]),
            ("지역",        f'{rec["province"]} {rec["district"]}'),
            ("요약",        rec["persona_summary"]),
            ("─" * 8,       "─" * 8),
            ("투표",        rec["vote"]),
            ("이유",        rec["reason"]),
            ("모델",        rec["model"]),
            ("버전",        version),
            ("소요시간",     f'{rec["elapsed_sec"]}s'),
            ("응답파일",     rec["response_file"]),
        ]
        width = max(len(k) for k, _ in labels)
        for k, v in labels:
            print(f"{k:<{width}}  {v}")
    return rec

def run_loop(provider: str, model_name: str | None, n: int, version: str = VERSION):
    """n회 반복. 매 호출 결과는 1줄 요약, 마지막 1건만 항목별 풀 출력. 에러는 건너뜀."""
    print(f"=== {provider}/{model_name or MODEL} × {n} (version={version}) ===")
    success = 0
    for i in range(n):
        rec = run_one(provider, model_name, version, verbose=(i == n - 1))
        if rec is None:
            continue
        success += 1
        if i < n - 1:
            print(f"[{i+1:>3}/{n}] {rec['sex']} {rec['age']}세 {rec['province']} "
                  f"{rec['occupation']:<12} → {rec['vote']:<10} ({rec['elapsed_sec']}s)")
    print(f"\n결과: {success}/{n} 성공")

In [69]:
# [Cell 9] OpenAI — 반복 회수 N + 버전 조정 후 실행
N = 70
VERSION_RUN = "v2"   # voter_context_*.md + system_prompt_*.md 같은 v로 페어링
run_loop("openai", MODEL, N, version=VERSION_RUN)

=== openai/gpt-5.4-mini × 70 (version=v2) ===
[  1/70] 여자 23세 경기 무직           → 더불어민주당     (5.041s)
[  2/70] 남자 75세 부산 무직           → 무당층/기권     (4.231s)
[  3/70] 남자 48세 서울 무직           → 무당층/기권     (2.455s)
[  4/70] 여자 48세 서울 무직           → 더불어민주당     (2.565s)
[  5/70] 남자 34세 광주 전자제품 개발 기술자 및 연구원 → 더불어민주당     (2.637s)
[  6/70] 여자 43세 부산 금형원          → 국민의힘       (4.14s)
[  7/70] 여자 35세 경기 무직           → 더불어민주당     (4.183s)
[  8/70] 남자 42세 서울 무직           → 무당층/기권     (4.16s)
[  9/70] 여자 65세 서울 그 외 철도 및 전동차 기관사 → 무당층/기권     (4.4s)
[ 10/70] 남자 46세 경기 해병대 부사관      → 더불어민주당     (2.2s)
[ 11/70] 여자 51세 경상북 수동 포장원       → 국민의힘       (4.07s)
[ 12/70] 여자 47세 서울 주방 보조원       → 더불어민주당     (2.476s)
[ 13/70] 남자 91세 대전 무직           → 국민의힘       (2.524s)
[ 14/70] 남자 64세 부산 무직           → 국민의힘       (4.242s)
[ 15/70] 여자 27세 전북 소방시설 공사 기술자 및 연구원 → 더불어민주당     (2.428s)
[ 16/70] 여자 57세 경상남 일반 비서        → 더불어민주당     (2.326s)
[ 17/70] 남자 64세 충청남 무직           → 국민의힘       (2.707s)
[ 18/70] 남자 60세 경기 전직 변호사,

In [66]:
# [Cell 10] EXAONE 4.0 32B (LG, 한국어 특화)
N = 0
VERSION_RUN = "v2"
run_loop("ollama", "ingu627/exaone4.0:32b", N, version=VERSION_RUN)

=== ollama/ingu627/exaone4.0:32b × 0 (version=v2) ===

결과: 0/0 성공


In [ ]:
# [Cell 11] Qwen3 32B (Apache 2.0, 다국어)
N = 0
VERSION_RUN = "v2"
run_loop("ollama", "qwen3:32b", N, version=VERSION_RUN)

=== ollama/qwen3:32b × 110 (version=v2) ===
[  1/110] 여자 25세 전라남 주방 보조원       → 더불어민주당     (38.842s)
[ERROR] (46.4s) missing keys: ['성별', '연령', '혼인상태', '가구형태', '주거', '학력', '전공', '직업', '지역', '지지정당', '추론근거']
[  3/110] 남자 43세 서울 무직           → 더불어민주당     (59.079s)
[ERROR] (63.2s) no parseable JSON in response: '```json\n{\n  "지지정당": "더불어민주당",\n  "이유": [\n    "이지현 씨는 20대 여성으로, 세대별 정치 성향에서 청년층의 진보 성향
[  5/110] 여자 74세 경상북 전화 상담원       → 국민의힘       (36.27s)
[  6/110] 여자 48세 서울 경리 사무원       → 더불어민주당     (53.682s)
[ERROR] (63.4s) no parseable JSON in response: ''
[  8/110] 남자 84세 서울 무직           → 국민의힘       (28.213s)
[  9/110] 여자 93세 충청남 무직           → 무당층/기권     (38.435s)
[ERROR] (55.5s) missing keys: ['투표정당', '이유']
[ERROR] (63.3s) no parseable JSON in response: ''
[ 12/110] 여자 69세 강원 건설 단순 종사원    → 보수 성향의 정당  (63.674s)
[ 13/110] 남자 45세 세종 플랜트공학 기술자 및 연구원 → 자유한국당      (47.042s)
[ 14/110] 여자 66세 서울 건물 내 전기 설치 및 정비원 → 국민의힘       (53.962s)
[ 15/110] 여자 35세 전북 일반 비서        → 더불어민주당     (41.127s)


In [ ]:
# [Cell 12] Gemma 3 27B (Google, 140개 언어)
N = 0
VERSION_RUN = "v2"
run_loop("ollama", "gemma3:27b", N, version=VERSION_RUN)

## 다음 단계

`sample-persona` ~ `save` 셀을 반복 실행하면 한 명씩 `vote_results_all.csv`에 누적됩니다.

- OpenAI 모델은 [Cell 1]의 `MODEL` 변수, 로컬 모델은 [Cell 10~12]의 model_name 인자에서 변경
- `temperature=0.7` 로 다양성 확보. 결정적 결과 원하면 `0`으로
- N=500/1000 배치는 별도 노트북에서 루프·재시도·층화추출 추가 예정